This notebook create features based on the processed dataset from the previous notebook

In [1]:
import numpy as np
import pandas as pd
import os

from toast_cap.utilities import config

import timeit

In [2]:
start_time = timeit.default_timer()

In [3]:
run_date_str = '20231010'
s3_output = f's3://toast-datascience-sandbox/PradeepA/pre90_pd_model/{run_date_str}' # output directory

df = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/pre90_pd_model/{run_date_str}/full_raw_data.parquet') # read 'full_raw_data.parquet' generated from the previous notebook
df_defaults = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/pre90_pd_model/{run_date_str}/tc_default.parquet') # read 'full_raw_data.parquet' generated from the previous notebook

# you can add some filters to reduce the dataset you're going to work with
df = df.loc[(~df['label_90'].isna()) & (df['dt'].dt.year>=2018)]
df = df.sort_values(by=['rid', 'dt'])

In [4]:
### I created a lot of features below, you may add or remove some features

In [5]:
# time on Toast
df['days_with_toast'] = ((df['dt'] - pd.to_datetime(df['first_order_date'])).dt.days).clip(lower=0)
df['months_with_toast'] = ((df.dt - df.first_order_date)/np.timedelta64(1, 'M'))
df['months_with_toast'] = df['months_with_toast'].astype(int)

In [6]:
# #Select 
# df = df[df['days_with_toast'] <= 90]

In [7]:
df['months_with_toast'].head()

365    54
366    54
367    54
368    54
369    54
Name: months_with_toast, dtype: int64

In [8]:
# GPV trend
# df['gpv_mean_365d'] = df.set_index('dt').groupby(
#     config.GROUP
# ).rolling('365d')['totalCreditCardRevenue'].mean().values
# df['gpv_mean_180d'] = df.set_index('dt').groupby(
#     config.GROUP
# ).rolling('180d')['totalCreditCardRevenue'].mean().values


df['gpv_mean_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['totalCreditCardRevenue'].mean().values

## volatility
# df['gpv_cv_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['totalCreditCardRevenue'].std().values / df['gpv_mean_180d'].values
df['gpv_cv_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['totalCreditCardRevenue'].std().values / df['gpv_mean_90d'].values

df['log_gpv'] = np.where(df['totalCreditCardRevenue']>0, np.log(df['totalCreditCardRevenue']+1), 0)
# df['log_gpv_std_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['log_gpv'].std().values
df['log_gpv_std_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['log_gpv'].std().values
df = df.drop(columns='log_gpv')


df['gpv_mean_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['totalCreditCardRevenue'].mean().values

df['gpv_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['totalCreditCardRevenue'].median().values

df['gpv_median_84d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('84d')['totalCreditCardRevenue'].median().values

df['gpv_median_28d_median_84d_diff'] = df['gpv_median_28d'] - df['gpv_median_84d']

df['gpv_median_28d_mean_90d_diff'] = df['gpv_median_28d'] - df['gpv_mean_90d']

df['gpv_median_28d_28ddiff']= df.groupby(config.GROUP)['gpv_median_28d'].diff(28).fillna(0).values
df['gpv_median_28d_84ddiff']= df.groupby(config.GROUP)['gpv_median_28d'].diff(84).fillna(0).values

/tmp/ipykernel_20497/3665880623.py:14: RuntimeWarning: invalid value encountered in divide
  df['gpv_cv_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['totalCreditCardRevenue'].std().values / df['gpv_mean_90d'].values


In [9]:
# GPV per hour trend
df['gpv_per_hour'] = np.where(
    df['tx_hours'] == 0, 0,
    df['totalCreditCardRevenue'] / df['tx_hours']
)

df['gpv_per_hour_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['gpv_per_hour'].median().values

df['gpv_per_hour_median_28d_28ddiff']= df.groupby(config.GROUP)['gpv_per_hour_median_28d'].diff(28).fillna(0).values


In [10]:
# days with no GPV
df['flag_no_gpv'] = np.where(df['totalCreditCardRevenue']<=0, 1, 0)

df['days_no_gpv_90d'] = df.set_index('dt').groupby(config.GROUP)\
    .rolling('90d')['flag_no_gpv'].sum().values
df['days_no_gpv_28d'] = df.set_index('dt').groupby(config.GROUP)\
    .rolling('28d')['flag_no_gpv'].sum().values

df['days_no_gpv_28d_28ddiff'] = df.groupby(config.GROUP)['days_no_gpv_28d'].diff(28).fillna(0).values

df = df.drop(columns='flag_no_gpv')

In [11]:
# transaction hours
df['tx_hours_mean_14d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('14d')['tx_hours'].mean().values
df['tx_hours_median_14d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('14d')['tx_hours'].median().values

df['tx_hours_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['tx_hours'].median().values

df['tx_hours_median_28d_28ddiff'] = df.groupby(config.GROUP)['tx_hours_median_28d'].diff(28).fillna(0).values

In [12]:
# modules
df['has_oo_mod'] = np.where(
    df['live_online_ordering_module_count'] > 0, 1, 0
)
df['has_gc_mod'] = np.where(
    df['live_gift_card_module_count'] > 0, 1, 0
)

In [13]:
# 30-day no processing history
df['noprocessing_last_90d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('60d')['noprocessing'].max().values  #rolling 150d because 150+30 = 180

In [14]:
end_time = timeit.default_timer()
print(f"The time taken to run the code is {end_time - start_time} seconds.")

The time taken to run the code is 1030.7114111459814 seconds.


In [15]:
start_time = timeit.default_timer()

# flag - check if within the last 7 days, there are at least 3 non consecutive days with greater than $50 in credit card sales.
def check_flag(amounts):
    return int((amounts >= 50).sum() >= 3)

# Add a column for the rolling window flag
df['flag_ge50_3d_over_7d'] = df.groupby(config.GROUP)['totalCreditCardRevenue'].rolling(window=7, min_periods=1).apply(check_flag).reset_index(level=0, drop=True).astype(int)

# df['flag_ge50_3d_over_7d'] = df.set_index('dt').groupby(config.GROUP)['totalCreditCardRevenue'].rolling(window=7, min_periods=1).apply(check_flag).reset_index(level=0, drop=True).astype(int)

end_time = timeit.default_timer()
print(f"The time taken to run the code is {end_time - start_time} seconds.")

The time taken to run the code is 10227.902436720004 seconds.


In [16]:
df['flag_ge50_3d_over_7d'].value_counts()

1    72212094
0     7593476
Name: flag_ge50_3d_over_7d, dtype: int64

In [18]:
df.to_parquet(os.path.join(s3_output, 'processed_train.parquet'))